# B 收尾验收结果

以下代码只读取保存的JSON，不加载FPGA，不启动DMA，也不恢复/复位硬件。可在普通Jupyter内核按Shift+Enter查看；长时测试的running状态不等于已经通过。

In [ ]:
from pathlib import Path
import json
_board_root = Path('/home/xilinx/jupyter_notebooks/pcb_b_closure_v032')
_candidates = [_board_root]
for _parent in (Path.cwd(), *Path.cwd().parents):
    _candidates += [_parent, _parent / 'pcb_b_closure_v032']
ROOT = next((_p for _p in _candidates if (_p / 'NEW_PACKAGE_MANIFEST.json').is_file() and (_p / 'evidence/closure_stability_20261101_32c30001/report.json').is_file()), None)
assert ROOT is not None, '请先进入解压后的pcb_b_closure_v032或其software目录，再运行本笔记本'

def read_run(folder, name='report.json'):
    return json.loads((ROOT / 'evidence' / folder / name).read_text(encoding='utf-8'))
profile = read_run('profile_run_20261101_32a10001', 'profile_report.json')
validation = read_run('closure_validate_20261101_32b20001')
stable = read_run('closure_stability_20261101_32c30001')
print('证据来源：PYNQ-Z2；实测批次日期：2026-11-01')
print('本笔记本只读已保存报告，不产生新的硬件测试。')
print('读取目录：', ROOT)
print('小测试状态：', profile['state'], profile['passed'])
print('66次DMA对照状态：', validation['state'], validation['passed'])
print('连续测试状态：', stable['state'], stable['passed'])
print('板端原时间：', stable['environment']['board_utc'], '（原样保留，可能未校准）')


## 同一批次30轮完整调用

单位毫秒。新服务保留严格检查，当前仍慢于本次CPU参考。

In [ ]:
assert validation['state'] == 'complete' and validation['passed'], '本轮对照尚未成功完成'
print(f"{'实现':<12} {'中位数ms':>12} {'P95ms':>12}")
for label in ('cpu', 'old', 'new'):
    row = validation['aggregate'][label]['outer_wall_seconds']
    print(f"{label:<12} {row['median'] * 1000:12.3f} {row['p95_linear'] * 1000:12.3f}")
old = validation['aggregate']['old']['outer_wall_seconds']['median']
new = validation['aggregate']['new']['outer_wall_seconds']['median']
print('新服务比旧服务耗时减少：', round((1 - new / old) * 100, 2), '%')
print('实际硬件传输数：', validation['validated_hardware_frames'])
print('新服务严格校验：', validation['candidate'])
print('输出所有权：', validation['records'][-1]['calls']['new']['output_ownership'])


## 连续运行证据

范围仅为B固定黄金图硬件路径；相机、定位、界面和整机30分钟需要C另行验收。

In [ ]:
print('状态：', stable['state'], '通过：', stable['passed'])
print('要求秒数：', stable.get('requested_duration_seconds'))
print('实际已记录秒数：', stable.get('elapsed_monotonic_seconds'))
print('已校验帧数：', stable.get('validated_hardware_frames'))
print('错误数：', stable.get('errors'))
print('初始内存：', stable.get('initial_memory'))
print('末次内存：', stable.get('final_memory', stable.get('current_memory')))
if stable['state'] == 'complete' and stable['passed']:
    print('最终状态：', stable['final_status'])
    print('停止/缓冲区状态：', stable['final_driver_safety'])
else:
    print('尚不能把运行中的计数作为最终通过证据。')
print('RSS与全局CMA有波动，不据此单独宣称不存在任何内存泄漏。')


## 电脑独立复核与剩余任务

复核文件另存pc_reviews，保持管理员生成的原始运行目录只读。

In [ ]:
for mode in ('profile', 'validate', 'stability'):
    path = ROOT / 'pc_reviews' / ('independent_' + mode + '_audit.json')
    if path.is_file():
        audit = json.loads(path.read_text(encoding='utf-8'))
        print(mode, '电脑证据审计通过：', audit.get('audit_passed'))
    else:
        print(mode, '电脑复核尚未归档到板卡')
print('人工收尾：归档已有点灯照片/视频；补原project_1记录；队友独立复现。')
print('团队事项：A处理已知SAD误报；C完善相机和整机验收。')
